## Setup

In [ ]:
import serial
import csv
import time
from datetime import datetime
from pathlib import Path
from scipy.signal import get_window

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
from scipy.special import gammaincc
from scipy.stats import norm
import numpy.fft as fft

import pyqtgraph as pg
from pyqtgraph.Qt import QtCore
from scipy.signal import welch
import padasip as pa

PORT = "COM8"
BAUD = 921600

LOAD_FROM_FILE = False
LOAD_FILE_PATH = Path("cross_sensor_variance_data/07_02_2026/run_15_45_20/sample1.csv")  # set to None to read from serial port

SENSORS = ['BPW34_1', 'BPW34_2', 'Inano_1', 'Inano_2']

ADC_MAX = 16383 #2^14 - 1
V_REF = 5 #confirmed

SAMPLES_PER_PACKET = 50
payload_dtype = np.dtype([
    ('timestamp', '<u4'),
    ('BPW34_1', '<u2'),
    ('BPW34_2', '<u2'),
    ('Inano_1', '<u2'),
    ('Inano_2', '<u2')
])

BYTES_PER_SAMPLE = payload_dtype.itemsize * SAMPLES_PER_PACKET
CARRIER_FREQ = 500  # carrier frequency in Hz
ser = None
reader = None

def fir_lowpass_filter(data, cutoff_hz, fs, num_taps=101):
    b = signal.firwin(num_taps, cutoff=cutoff_hz, fs=fs, window='hamming')
    filtered_data = np.asarray(signal.lfilter(b, 1.0, data))
    return filtered_data, b

def adc_to_mv(adc_value):
    return (adc_value / ADC_MAX) * V_REF * 1000  # Convert to millivolts

def num_samples_in_file(file_path):
    df = pd.read_csv(file_path)
    times = df['timestamp'].to_numpy()
    return len(times)

def fmt_matrix(M, title, fmt="{:>12.4f}"):
    s = f"\n{title}\n"
    s += " " * 10 + "".join(f"{ch:>12}" for ch in SENSORS) + "\n"
    for i, ch in enumerate(SENSORS):
        s += f"{ch:>10}" + "".join(fmt.format(M[i, j]) for j in range(len(SENSORS))) + "\n"
    return s

# reads 50 samples from serial port.
def try_read_realtime(ser):
    if ser.read(1) == b'\xAA' and ser.read(1) == b'\xBB':
        raw_data = ser.read(BYTES_PER_SAMPLE) 
        if len(raw_data) == BYTES_PER_SAMPLE:
            data = np.frombuffer(raw_data, dtype=payload_dtype)
            dts = np.diff(data['timestamp'].astype(np.int64))
            ok = (all((data[s] <= ADC_MAX).all() for s in SENSORS) and
                  (dts > 0).all() and (dts < 5000).all())
            if ok:
                return data
    return None

# read 50 samples from file to emulate realtime reading.
def read_from_file_realtime(file_path):
    df = pd.read_csv(file_path)
    data = df[list(payload_dtype.names or ())].to_records(index=False).astype(payload_dtype)
    n_packets = len(data) // SAMPLES_PER_PACKET
    packets = [data[i * SAMPLES_PER_PACKET:(i + 1) * SAMPLES_PER_PACKET]
               for i in range(n_packets)]

    idx = 0
    ts_start = int(packets[0]['timestamp'][0]) if packets else 0  # first timestamp (us)
    wall_start = None  # wall-clock time

    def reader(*_): 
        nonlocal idx, wall_start
        if idx >= len(packets):
            return np.zeros(SAMPLES_PER_PACKET, dtype=payload_dtype)  # exhausted
        pkt = packets[idx]
        now = time.time()
        if wall_start is None:  # start the clock on the first call
            wall_start = now
        # only release the packet once real time has caught up to its timestamp
        elapsed_us = (now - wall_start) * 1e6
        if elapsed_us < int(pkt['timestamp'][0]) - ts_start:
            return None
        idx += 1
        return pkt

    return reader

def read_from_file_instant(file_path):
    df = pd.read_csv(file_path)
    data = df.to_records(index=False).astype(payload_dtype)
    return data


def save_data(samples_total, savefile_path, ser):
    savefile_path.parent.mkdir(parents=True, exist_ok=True)

    chunks = []
    sample_count = 0

    timeout = 0
    try:
        while samples_total is None or sample_count < samples_total:
            if timeout > 30:
                print("Timeout exceeded, stopping capture.")
                break
            if ser.in_waiting < BYTES_PER_SAMPLE + 2:
                continue
            data = try_read_realtime(ser)
            if data is None:
                timeout += 1
                continue 
            if np.mean(np.diff(data['timestamp'])) <= 0:
                print("Timestamps did not change")
                continue
            chunks.append(data)
            sample_count += SAMPLES_PER_PACKET
    except KeyboardInterrupt:
        print("Stopping...")
    finally:
        if chunks:
            df = pd.DataFrame(np.concatenate(chunks))
            df.to_csv(savefile_path, index=False) 
            print(f"Saved {sample_count} samples to {savefile_path.as_posix()}")
        else:
            print("No data captured.")

start = time.time()
time_chunks = []
full_data = {s: [] for s in SENSORS}
Fs = 1500.0  # initial guess for sampling rate
ser = serial.Serial(PORT, BAUD, timeout=1) if not LOAD_FROM_FILE else None

if LOAD_FROM_FILE:
    reader = read_from_file_realtime(LOAD_FILE_PATH)

while time.time() - start < 2:
    if LOAD_FROM_FILE:
        assert reader is not None 
        data = reader()
    else:
        data = try_read_realtime(ser)
    if data is not None:
        time_chunks.append(data['timestamp'])
        for s in SENSORS:
            full_data[s].append(adc_to_mv(data[s]))
if ser is not None and ser.is_open:
    ser.close()

Fs = float(np.mean([1.0 / (np.median(np.diff(time_chunks[i])) * 1e-6) for i in range(len(time_chunks))]))
print(f"Estimated sampling rate: {Fs:.2f} Hz")

estimated_carrier_freqs = {}

for s in SENSORS:
    x = np.concatenate(full_data[s])  # full_data[s] is a list of per-packet chunks
    fft_vals = fft.fft(x)
    f = fft.fftfreq(len(x), d=1/Fs)
    mags = 2 * np.abs(fft_vals) / len(x)
    inband = (f > 14)  # skip the DC / drift / 1-f region
    peaks, _ = signal.find_peaks(mags[inband], distance=5, height=0.1)  # adjust height threshold as needed
    sorted_peaks = sorted(peaks, key=lambda p: mags[inband][p], reverse=True)
    estimated_carrier_freqs[s] = f[inband][sorted_peaks[0]] if sorted_peaks else 0.0

CARRIER_FREQ = np.mean(list(estimated_carrier_freqs.values()))
print(f"Average estimated carrier frequency across sensors: {CARRIER_FREQ:.6f} Hz")

## Saves realtime data to file


In [ ]:

ser = None
if not LOAD_FROM_FILE:
    ser = serial.Serial(PORT, BAUD, timeout=1)     
    NUM_SAMPLES_TO_SAVE = 5000   # set to None to run until interrupted

    now = datetime.now()
    savefile_path = Path(f"data/{now.strftime('%m_%d_%Y')}/run_{now.strftime('%H_%M_%S')}.csv")

    save_data(NUM_SAMPLES_TO_SAVE, savefile_path, ser)

if ser is not None and ser.is_open:
    ser.close()

## Realtime plot

Reads packets from the Arduino and draws them sweep-style (like an ECG monitor): each incoming 50-sample packet is written at the cursor — the blank gap in the trace — and everything already drawn stays put until the cursor wraps around and overwrites it.

In [ ]:

ser = serial.Serial(PORT, BAUD, timeout=1) if not LOAD_FROM_FILE else None
if ser is not None:
    ser.reset_input_buffer()  # drop bytes buffered before this cell started so we begin on live data


N = 1000
assert N % SAMPLES_PER_PACKET == 0
buffers = {s: np.full(N, np.nan) for s in SENSORS}  # nan = not drawn (unfilled region / cursor gap)
write_idx = 0

FILTER = False

label_style = {'color': '#FFF', 'font-size': '16pt'}

app = pg.mkQApp("realtime")
win = pg.GraphicsLayoutWidget(show=True, title="realtime")
pen = pg.mkPen('r', width=5)
curves = {}
for i, s in enumerate(SENSORS):
    p = win.ci.addPlot(row=i, col=0)
    p.setLabel('left', 'mV', **label_style)
    p.setLabel('top', f'{s}', **label_style)
    curves[s] = p.plot(pen=pen, connect='finite')  # 'finite' leaves a gap at the nan cursor instead of a line across it

sos_bp = signal.butter(4, [CARRIER_FREQ - 2, CARRIER_FREQ + 2], btype='bp', fs=Fs, output='sos')
sos_zi = {s: signal.sosfilt_zi(sos_bp) for s in SENSORS}

if LOAD_FROM_FILE:
    reader = read_from_file_realtime(LOAD_FILE_PATH)

def update():
    # Sweep display: each new packet overwrites the 50 samples at the cursor and
    # nothing else moves, so previously plotted values never change.
    global write_idx
    got_packet = False

    while True:
        if LOAD_FROM_FILE:
            data = reader()
            if data is None:  # reader is pacing itself against wall clock
                break
        else:
            if ser is not None and ser.in_waiting < BYTES_PER_SAMPLE + 2:
                break  # no complete packet buffered; never block the GUI thread
            data = try_read_realtime(ser)
            if data is None:
                continue  # header scan slipped a byte or two; keep scanning the backlog
        for s in SENSORS:
            if FILTER:
                cleaned, sos_zi[s] = signal.sosfilt(sos_bp, data[s], zi=sos_zi[s])
                buffers[s][write_idx:write_idx + SAMPLES_PER_PACKET] = adc_to_mv(cleaned)
            else:
                buffers[s][write_idx:write_idx + SAMPLES_PER_PACKET] = adc_to_mv(data[s])
        write_idx = (write_idx + SAMPLES_PER_PACKET) % N
        got_packet = True
        if LOAD_FROM_FILE:
            break  
    if got_packet:
        for s in SENSORS:
            buffers[s][write_idx:write_idx + SAMPLES_PER_PACKET] = np.nan  # blank slot marks the cursor
            curves[s].setData(buffers[s])

timer = QtCore.QTimer()
timer.timeout.connect(update)
timer.start(10)
pg.exec()
timer.stop()  # don't leave a stale timer attached if the cell is re-run
if ser is not None and ser.is_open:
    ser.close()

## Spectrogram Plotting

Make sure to specify load_from_file in the setup block to load from file instead of read real time data

In [ ]:
from scipy.signal import get_window

ser = None
if not LOAD_FROM_FILE:
    ser = serial.Serial(PORT, BAUD, timeout=1)

NFFT = 4096        
WINDOWSIZE = 512
WINDOW = 'hamming'
OVERLAP = 90 
FILTER = False
CAPTURE_TIME = 9  # seconds

## load from file or read from serial
## after this block: samples[ch] and times are plain 1-D numpy arrays
if LOAD_FROM_FILE:
    reader = read_from_file_realtime(LOAD_FILE_PATH)
    

sample_chunks = {s: [] for s in SENSORS}
time_chunks = []
start = time.time()

timeout = 0
# !!! ENSURE DATA IN FILE IS ATLEAST AS LONG AS CAPTURE TIME
while time.time() - start < CAPTURE_TIME:
    if timeout > 30:
        print("Timeout exceeded, stopping capture.")
        break
    if LOAD_FROM_FILE:
        data = reader()
    else:
        data = try_read_realtime(ser)
    if data is not None:
        time_chunks.append(data['timestamp'])
        for s in SENSORS:
            sample_chunks[s].append(adc_to_mv(data[s]))
    else:
        timeout += 1
if ser is not None and ser.is_open:
    ser.close()

samples = {s: np.concatenate(sample_chunks[s]) for s in SENSORS}
times = np.concatenate(time_chunks)

win = get_window(WINDOW, WINDOWSIZE)
noverlap = int(WINDOWSIZE * OVERLAP / 100)

sos_bp = signal.butter(4, [CARRIER_FREQ - 2, CARRIER_FREQ + 2], btype='bp', fs=Fs, output='sos') 
filtered = {s: signal.sosfilt(sos_bp, samples[s]) for s in SENSORS}

with plt.style.context('dark_background'):
    fig, axes = plt.subplots(4, 1, figsize=(19, 21), sharex=False, constrained_layout=True)
    fig.suptitle(f'Channel spectrograms, Sampling Rate = {Fs:.2f} Hz', fontsize=30)

    for ax, s in zip(axes, SENSORS):
        if FILTER:
            spec, freqs, t_bins, im = ax.specgram(filtered[s], Fs=Fs, NFFT=WINDOWSIZE, window=win, noverlap=noverlap, pad_to=NFFT, cmap='magma')
        else:
            spec, freqs, t_bins, im = ax.specgram(samples[s], Fs=Fs, NFFT=WINDOWSIZE, window=win, noverlap=noverlap, pad_to=NFFT, cmap='magma')
        ax.set_ylabel(f'{s}\n(Hz)', fontsize = 30)
        ax.set_ylim(0, Fs/2)
        cbar = fig.colorbar(im, ax=ax, pad=0.01)
        cbar.set_label('Power (dB)', fontsize = 30)

    axes[-1].set_xlabel('Time (s)', fontsize = 30)
    plt.show()


## Realtime PSD

In [ ]:
ser = None
if not LOAD_FROM_FILE:
    ser = serial.Serial(PORT, BAUD, timeout=1)

FILTER = True # set to True to enable bandpass filtering

N = 2000
buffers = {s: np.zeros(N) for s in SENSORS}

sos_bp = signal.butter(4, [CARRIER_FREQ - 2, CARRIER_FREQ + 2], btype='bp', fs=Fs, output='sos')
sos_zi = {s: signal.sosfilt_zi(sos_bp) for s in SENSORS}

app = pg.mkQApp("QRNG PSD realtime")
win = pg.GraphicsLayoutWidget(show=True, title="QRNG PSD realtime")
curves = {}
plots = []
xticks = [[(1, '1')] + [(f, str(f)) for f in range(100, 1001, 100)]]  # 1, 100, 200, ... 1000
for i, s in enumerate(SENSORS):
    p = win.ci.addPlot(row=0, col=i, title=s)
    p.setLabel('left', 'PSD (mV^2/Hz)')
    p.setLabel('bottom', 'Frequency (Hz)')
    p.getViewBox().setXRange(1, Fs, padding=0)
    p.getAxis('bottom').setTicks(xticks)  # force 1, 100, 200, ... 1000 labels
    plots.append(p)
    curves[s] = p.plot()

if LOAD_FROM_FILE:
    reader = read_from_file_realtime(LOAD_FILE_PATH)

def update():
    if LOAD_FROM_FILE:
        data = reader()
    else:
        data = try_read_realtime(ser)
    if data is not None:
        for s in SENSORS:
            chunk_clean = (data[s] / ADC_MAX * V_REF)*1000 ##from v to mV
            # bandpass this chunk, carrying filter state across chunks
            chunk, sos_zi[s] = signal.sosfilt(sos_bp, chunk_clean, zi=sos_zi[s])
            buffers[s] = np.roll(buffers[s], -SAMPLES_PER_PACKET)
            if FILTER:
                buffers[s][-SAMPLES_PER_PACKET:] = chunk
            else:
                buffers[s][-SAMPLES_PER_PACKET:] = chunk_clean
            f, pxx = welch(buffers[s], fs=Fs, nperseg=512)
            curves[s].setData(f, pxx)

timer = QtCore.QTimer()
timer.timeout.connect(update)
timer.start()
pg.exec()
if ser is not None and ser.is_open:
    ser.close()

## Variance of A Single Sensor over Sequential Time Periods

In [ ]:
ser = None
if not LOAD_FROM_FILE:
    ser = serial.Serial(PORT, BAUD, timeout=1)

LOAD_FILE_PATHS = [
    Path("VarianceData/06_30_2026/run_11_04_33/sample1.csv"),
    Path("VarianceData/06_30_2026/run_11_04_33/sample2.csv"),
    Path("VarianceData/06_30_2026/run_11_04_33/sample3.csv"),
    Path("VarianceData/06_30_2026/run_11_04_33/sample4.csv"),]

NUM_SAMPLES_TO_SAVE = 5000   # set to None to run until interrupted
SENSOR = SENSORS[0]
NUM_TRIALS = 6
PERIODS = [f"t{i+1}" for i in range(NUM_TRIALS)]
TRANSIENT = 200  # samples to drop to ensure filter settles
now = datetime.now()
savefolder_path = Path(f"temporal_variance_data/{now.strftime('%m_%d_%Y')}/run_{now.strftime('%H_%M_%S')}")

data = {}

#save 4 sets of data to file, then run this analysis on the saved file.  

if not LOAD_FROM_FILE:
    for i in range(NUM_TRIALS):
        savefile_path = savefolder_path / f"sample{i+1}.csv"
        save_data(NUM_SAMPLES_TO_SAVE, savefile_path, ser)
        rec = read_from_file_instant(savefile_path)      # one structured array (all sensor columns)
        data[i] = {s: rec[s] for s in SENSORS}      # split into one sample array per sensor
    if ser is not None and ser.is_open:
        ser.close()
else:
    for i in range(NUM_TRIALS):
        rec = read_from_file_instant(LOAD_FILE_PATHS[i])      # one structured array (all sensor columns)
        data[i] = {s: rec[s] for s in SENSORS}      # split into one sample array per sensor


# only look at SENSOR
data = {i: adc_to_mv(data[i][str(SENSOR)]) for i in range(NUM_TRIALS)}
times = {i: data[i]['timestamp'] for i in range(NUM_TRIALS)}

Fs = np.mean([1.0 / (np.median(np.diff(times[i])) * 1e-6) for i in range(NUM_TRIALS)])

sos_bp = signal.butter(4, [CARRIER_FREQ - 2, CARRIER_FREQ + 2], btype='bp', fs=Fs, output='sos')
sos_lp = signal.butter(4, 150, btype='low', fs=Fs, output='sos')

filtered_data = {i: signal.sosfiltfilt(sos_bp, data[i]) for i in range(NUM_TRIALS)}
kept_filtered_data = {i: filtered_data[i][TRANSIENT:-TRANSIENT] for i in range(NUM_TRIALS)}


n = min(len(kept_filtered_data[i]) for i in range(NUM_TRIALS))
t = np.arange(n) / Fs 

A = {}
a = {}
for i in range(NUM_TRIALS):
    x = kept_filtered_data[i][:n]
    I = np.cos(2 * np.pi * CARRIER_FREQ * t) * x
    Q = -np.sin(2 * np.pi * CARRIER_FREQ * t) * x
    I_f = signal.sosfiltfilt(sos_lp, I)
    Q_f = signal.sosfiltfilt(sos_lp, Q)
    A[i] = (2 * np.sqrt(I_f**2 + Q_f**2))[TRANSIENT:-TRANSIENT]
    a[i] = np.mean(A[i])

a_vec = np.array([a[i] for i in range(NUM_TRIALS)])

var_A = np.array([np.var(A[i]) for i in range(NUM_TRIALS)])  # mV^2, total fluctuation per window
std_A = np.sqrt(var_A) # mV
frac  = std_A / a_vec # std/mean: fractional fluctuation (m + n combined)

result = f"A is the message encoded in the carrier wave (the total noise signal, LED FLUCTUATION + CIRCUIT NOISE + INTERFERENCE...)\n" 
result += f"a is the mean of the demodulated message\n"
result += f"Var(A) is the total variance of A\n"
result += f"std(A) is the standard deviation of A\n"
result += f"std/a is the fractional fluctuation (std/mean) for each time window.\n"

result += f"\nSame sensor ({SENSOR}) over {NUM_TRIALS} sequential time windows, Fs = {Fs:.2f} Hz\n"
result += f"{'window':>8}{'a (mV)':>12}{'Var(A) mV^2':>14}{'std(A) mV':>12}{'std/a':>12}\n"
for i in range(NUM_TRIALS):
    result += f"{PERIODS[i]:>8}{a_vec[i]:>12.4f}{var_A[i]:>14.4f}{std_A[i]:>12.4f}{frac[i]:>12.4e}\n"

# across-window summary: the spread of Var(A) shows how stationary the total
# fluctuation is. CV = std/mean of Var(A) across windows; small => stationary.
mean_var = var_A.mean()
std_var  = var_A.std(ddof=1)
cv = std_var / mean_var

result += f"\nVar(A) across windows: E[Var(A)] = {mean_var:.4f} mV^2, std = {std_var:.4f} mV^2, spread (CV) = {100*cv:.1f}%"
print(result)
with open(savefolder_path / "summary.txt", "w") as f:
    f.write(result + "\n")





## Find the Variance Between Sensors

$f_c$ = 500Hz

The measured voltage over the load resistor due to the current of a photodiode in response to an LED at $f_c$: $V(t) = A(t)cos(2 \pi f_c t + \theta) $  

$A(t) = a + am(t) + n(t)$ where

a = the mean of the carrier amplitude, the voltage due to photocurrent, mV 

m(t) = the LED relative intensity fluctuation (common to all of the PDS), dimensionless, E[m] = 0

n(t) = the indepedent noise of each PD (shot, thermal, etc) within the band, mV

By using IQ Demodulation, A(t) can be recovered from the carrier wave.

The covariance of all of the A signal pairs from each diode pair can be found as 

$Cov(A_1, A_2) = a_1a_2Var(m)$ since n(t) is uncorrelated and all PDs see the same fractional fluctuation from the LED.  Thus, $Var(m)$ can be isolated as $Var(m) = \frac{Cov(A_1,A_2)}{a_1a_2}$.



Var(m) can be estimated by using the above formula and taking the mean of the most correlated pairs.

Since n(t) is uncorrelated, it can be found that $Var(n_i) = Var(A_i) - a_i^2Var(m)$ for each PD.

In [ ]:
ser = None
if not LOAD_FROM_FILE:
    ser = serial.Serial(PORT, BAUD, timeout=1)

LOAD_FILE_PATHS = []

NUM_SAMPLES_TO_SAVE = 40000   # set to None to run until interrupted
NUM_TRIALS = 2
CORR_GATE = 0.5  # only average pairs that actually share the common mode
PERIODS = [f"t{i+1}" for i in range(NUM_TRIALS)]
TRANSIENT = 200  # samples to drop on each end to ensure filter settles
now = datetime.now()
savefolder_path = Path(f"cross_sensor_variance_data/{now.strftime('%m_%d_%Y')}/run_{now.strftime('%H_%M_%S')}")

# data[trial] -> {sensor: 1-D array of that sensor's raw ADC samples for the trial}
data = {}

if not LOAD_FROM_FILE:
    for i in range(NUM_TRIALS):
        savefile_path = savefolder_path / f"sample{i+1}.csv"
        save_data(NUM_SAMPLES_TO_SAVE, savefile_path, ser)
        rec = read_from_file_instant(savefile_path)      
        data[i] = {s: rec[s] for s in SENSORS}   
    if ser is not None and ser.is_open:
        ser.close()
else:
    for i in range(NUM_TRIALS):
        rec = read_from_file_instant(LOAD_FILE_PATHS[i])
        data[i] = {s: rec[s] for s in SENSORS}


# filters:
# sos_bp: isolate the carrier (CARRIER_FREQ +/- 2 Hz)
# sos_lp: extract the demodulated envelope after mixing the carrier down to baseband 
# Both zero-phase (filtfilt)
sos_bp = signal.butter(4, [CARRIER_FREQ - 2, CARRIER_FREQ + 2], btype='bp', fs=Fs, output='sos')
sos_lp = signal.butter(4, 150, btype='low', fs=Fs, output='sos')

offdiag = ~np.eye(len(SENSORS), dtype=bool)   # mask that selects off-diagonal (cross) entries


def iq_demodulate(carrier_signal, t):
    I = signal.sosfiltfilt(sos_lp,  np.cos(2 * np.pi * CARRIER_FREQ * t) * carrier_signal)
    Q = signal.sosfiltfilt(sos_lp, -np.sin(2 * np.pi * CARRIER_FREQ * t) * carrier_signal)
    return 2.0 * np.sqrt(I ** 2 + Q ** 2)


result = "A_i is the demodulated message of sensor i (LED FLUCTUATION + CIRCUIT NOISE + INTERFERENCE...)\n"
result += "a_i is the mean carrier amplitude of sensor i\n"
result += "Cov(A_i, A_j) = a_i * a_j * Var(m)  ->  Var(m) = Cov(A_i, A_j) / (a_i * a_j)\n"
result += "Var(n_i) = Var(A_i) - a_i^2 * Var(m)  (independent per-diode noise)\n"
result += f"\nAcross {len(SENSORS)} sensors over {NUM_TRIALS} independent windows, Fs = {Fs:.2f} Hz, Carrier Frequency = {CARRIER_FREQ:.2f} Hz\n"

var_m_trials = []

for trial in range(NUM_TRIALS):
    # raw ADC -> mV -> bandpass around the carrier, then drop the filter transient
    bandpassed = {}
    for s in SENSORS:
        v_mv = adc_to_mv(data[trial][s])
        v_bp = signal.sosfiltfilt(sos_bp, v_mv)
        bandpassed[s] = v_bp[TRANSIENT:-TRANSIENT]

    n = min(len(bandpassed[s]) for s in SENSORS)   # common length across channels
    t = np.arange(n) / Fs

    # recover each channel's envelope A_i(t) and its mean amplitude a_i
    # model:  A_i(t) = a_i + a_i*m(t) + n_i(t)
    A = {}
    a = {}
    for s in SENSORS:
        envelope = iq_demodulate(bandpassed[s][:n], t)
        A[s] = envelope[TRANSIENT:-TRANSIENT] # drop the low-pass edge transient
        a[s] = A[s].mean()

    X = np.vstack([A[s] for s in SENSORS])
    a_vec  = np.array([a[s] for s in SENSORS])
    cov_A  = np.cov(X) # mV^2, Var(A_i) on the diagonal
    corr_A = np.corrcoef(X)  # dimensionless, in [-1, 1]
    norm_cov = cov_A / np.outer(a_vec, a_vec) # Cov(A_i,A_j) / (a_i * a_j)

    # mask out entries that aren't correlated; var(m) = mean of what survives (nan if none)
    gated = np.where(offdiag & (corr_A > CORR_GATE), norm_cov, np.nan)
    var_m = np.nanmean(gated)
    n_pairs = int(np.isfinite(gated).sum() // 2)
    is_coupled = np.isfinite(gated).sum(axis=1) > 0
    var_m_trials.append(var_m)


    var_A = np.diag(cov_A)  # total envelope variance per channel
    var_common = a_vec ** 2 * var_m # portion explained by the shared LED
    var_n = var_A - var_common # remainder = independent, uncorrelated noise

    result += f"\n{'=' * 64}\nTRIAL {PERIODS[trial]}\n{'=' * 64}\n"
    result += fmt_matrix(cov_A, "[1] COVARIANCE (mV^2) - diagonal = total fluctuations of each channel, off-diagonal = fluctuation shared between two channels")
    result += fmt_matrix(corr_A, "[2] CORRELATION (-1 to 1)-  near 1 = channels track togther (shared LED), 0 = independent")
    result += fmt_matrix(norm_cov, "[3] NORMALIZED COVARIANCE - off-diagonals = var(m) estimate per pair; all equal => one shared LED fluctuation", fmt="{:>12.4e}")
    result += f"\nvar(m) = {var_m:.4e}   RMS(m) = sqrt(var(m)) = {np.sqrt(var_m):.4f}"
    result += f"   (mean over {n_pairs} coupled pair(s), corr > {CORR_GATE})\n"

    # per-channel noise split:  Var(A_i) = a_i^2*Var(m) [common] + Var(n_i) [independent]
    result += f"\n{'channel':>10}{'a_i (mV)':>10}{'Var(A_i)':>12}{'a^2*Var(m)':>12}{'Var(n_i)':>12}{'% indep':>9}{'coupled':>9}\n"
    clamped_any = False
    for i, s in enumerate(SENSORS):
        vn = var_n[i]
        if vn < 0: # unphysical: a_i^2*Var(m) > Var(A_i)
            vn = 0.0  # catastrophic cancellation -> report as ~0
            clamped_any = True
        pct_indep = 100 * vn / var_A[i]
        flag = "yes" if is_coupled[i] else "NO"
        result += (f"{s:>10}{a_vec[i]:>10.3f}{var_A[i]:>12.4f}" f"{var_common[i]:>12.4f}{vn:>12.4f}{pct_indep:>8.1f}%{flag:>9}\n")
        #result += f"The LED fluctuations make up {100 * a_vec[s] ** 2 * var_m / var_A[s]:.1f}% of the total variance of sensor {s}.\n"
    if clamped_any:
        result += "  Var(n_i)=0 (clamped): a_i^2*Var(m) >= Var(A_i), i.e. ~100% common mode; independent noise is below this method's resolution.\n"
    

result += f"\nE[RMS(m)] over {NUM_TRIALS} trial(s) = {np.sqrt(np.nanmean(var_m_trials)):.4f}  (fractional LED intensity fluctuation)\n"

result += f"\n{'=' * 64}\nINTERPRETATION\n{'=' * 64}\n"
result += (
    "Here are the variables in the model V(t) = A_i(t)cos(2πf_c t + θ),  A_i(t) = a_i + a_i·m(t) + n_i(t):\n\n"
    "Measured / raw\n"
    "    V_i(t)      — the raw load voltage of diode i (the carrier sine you actually sample), mV.\n"
    "    f_c         — carrier frequency, ~287 Hz (LED drive tone). θ is its phase.\n\n"
    "Per-diode envelope (after I/Q demod)\n"
    "    A_i(t)      — demodulated envelope of diode i: the slowly-varying carrier amplitude, mV. "
    "This is what all the covariance math runs on.\n"
    "    a_i         — mean carrier amplitude of diode i (the constant part of A_i), mV. "
    "Represents the average optical power reaching that diode. ~8–18 mV now.\n\n"
    "The two things we're separating\n"
    "    m(t)        — LED's fractional intensity fluctuation, dimensionless, E[m]=0. "
    "Shared by all diodes (common mode). RMS(m) ≈ 1–2% = how much the brightness wobbles relative to its mean.\n"
    "    n_i(t)      — independent per-diode in-band noise (shot/thermal + local pickup), mV. "
    "Uncorrelated across diodes. Currently pinned at the electronics floor.\n\n"
    "Computed statistics\n"
    "    Var(m)      — variance of m, dimensionless. Estimated as Cov(A_i,A_j)/(a_i·a_j).\n"
    "    RMS(m)      — √Var(m), the fractional fluctuation (0.01 = 1%). E[RMS(m)] averages it over trials.\n"
    "    Var(A_i)    — total variance of diode i's envelope, mV² (diagonal of cov_A).\n"
    "    a_i²·Var(m) — the part of Var(A_i) explained by the shared LED (common mode), mV².\n"
    "    Var(n_i)    — remainder Var(A_i) − a_i²·Var(m) = independent noise, mV².\n"
    "    cov_A       — envelope covariance matrix, mV² (Var(A_i) on diagonal, shared fluctuation off-diagonal).\n"
    "    corr_A      — envelope correlation matrix, −1…1 (near 1 = tracking the same LED).\n"
    "    norm_cov    — cov_A / (a_i·a_j); off-diagonals are the per-pair Var(m) estimate.\n"
)

print(result)

if not LOAD_FROM_FILE:
    savefolder_path.mkdir(parents=True, exist_ok=True)
    with open(savefolder_path / "summary.txt", "w", encoding='utf-8') as f:
        f.write(result + "\n")

if ser is not None and ser.is_open:
    ser.close()

## Signal Information

Sanity checks behind the variance analysis (run after Setup, points at the same CSVs as the variance cell):

1. **Carrier freq** — fine FFT peak vs the Welch-binned `CARRIER_FREQ` (off-center LO fakes AM on the filter skirt)
2. **Clock lock** — carrier phase wander vs the ADC timeline (two free-running boards showed 0.12 rad; one board 0.004 rad)
3. **AM/PM split** — is the fluctuation on the amplitude axis (what `2√(I²+Q²)` and the covariance model see) or in quadrature?
4. **Cross-channel corr** — complex-baseband corr (full common mode) vs envelope corr (AM part only)
5. **Additive floor** — demod a carrier-free band; predicts the amplitude-independent part of `Var(A)`/`Var(n_i)`
6. **Bandwidth sweep** — floor-dominated envelopes grow ∝ bandwidth
7. **CMR preview** — `d = A₁/a₁ − A₂/a₂`: what survives common-mode rejection, and is it drift or broadband

Summary text is saved to `signal_diagnostics_data/<date>/run_<time>/summary.txt`.

In [ ]:
ser = None
if not LOAD_FROM_FILE:
    ser = serial.Serial(PORT, BAUD, timeout=1)

LOAD_FILE_PATHS = [
    Path("cross_sensor_variance_data/07_02_2026/run_15_23_31/sample1.csv"),
    Path("cross_sensor_variance_data/07_02_2026/run_15_23_31/sample2.csv"),]

NUM_SAMPLES_TO_SAVE = 10000   # set to None to run until interrupted
NUM_TRIALS = 2
TRANSIENT = 200  # samples to drop on each end to ensure filter settles
BP_HALF_WIDTH = 2.0           # analysis band, same as the variance cell
EMPTY_BAND_FREQ = 279.0       # carrier-free band used to measure the additive electronics floor
SWEEP_HALF_WIDTHS = [0.5, 1.0, 2.0, 5.0, 10.0]
PERIODS = [f"t{i+1}" for i in range(NUM_TRIALS)]
now = datetime.now()
savefolder_path = Path(f"signal_diagnostics_data/{now.strftime('%m_%d_%Y')}/run_{now.strftime('%H_%M_%S')}")

# data[trial] -> {sensor: 1-D array of that sensor's raw ADC samples for the trial}
data = {}

if not LOAD_FROM_FILE:
    for i in range(NUM_TRIALS):
        savefile_path = savefolder_path / f"sample{i+1}.csv"
        save_data(NUM_SAMPLES_TO_SAVE, savefile_path, ser)
        rec = read_from_file_instant(savefile_path)
        data[i] = {s: rec[s] for s in SENSORS}
    if ser is not None and ser.is_open:
        ser.close()
else:
    for i in range(NUM_TRIALS):
        rec = read_from_file_instant(LOAD_FILE_PATHS[i])
        data[i] = {s: rec[s] for s in SENSORS}

sos_lp = signal.butter(4, 150, btype='low', fs=Fs, output='sos')


def complex_baseband(v_mv, fc, half_width):
    # bandpass around fc, mix to baseband: z = I + jQ. Envelope = 2|z|, so envelope-unit
    # variances below are 4x the variances of z's parts (comparable to Var(A) in the variance cell).
    sos_bp = signal.butter(4, [fc - half_width, fc + half_width], btype='bp', fs=Fs, output='sos')
    x = signal.sosfiltfilt(sos_bp, v_mv)[TRANSIENT:-TRANSIENT]
    t = np.arange(len(x)) / Fs
    I = signal.sosfiltfilt(sos_lp,  np.cos(2 * np.pi * fc * t) * x)
    Q = signal.sosfiltfilt(sos_lp, -np.sin(2 * np.pi * fc * t) * x)
    return (I + 1j * Q)[TRANSIENT:-TRANSIENT]


def lowfreq_fraction(x, f_edge):
    # fraction of variance below f_edge via PSD integration (a sub-Hz Butterworth
    # high-pass at this Fs is numerically unstable, so don't split by filtering)
    dec = x - x.mean()
    for _ in range(2):  # decimate 100x in two stages to keep filtfilt pad < signal length
        dec = signal.decimate(dec, 10, ftype='fir', zero_phase=True)
    f, P = welch(dec, fs=np.float64(Fs / 100), nperseg=min(len(dec), 2048))
    return np.trapezoid(P[f <= f_edge], f[f <= f_edge]) / np.trapezoid(P, f)


result = "SIGNAL-PATH DIAGNOSTICS - is the common mode really AM (what the covariance model needs)?\n"
result += "[1] carrier freq  [2] clock lock  [3] AM/PM split  [4] cross-channel corr  [5] additive floor  [6] bandwidth sweep  [7] CMR preview\n"
result += f"\n{len(SENSORS)} sensors x {NUM_TRIALS} windows, Fs = {Fs:.2f} Hz, LO = CARRIER_FREQ = {CARRIER_FREQ:.3f} Hz, band = +/-{BP_HALF_WIDTH} Hz\n"

for trial in range(NUM_TRIALS):
    mv = {s: adc_to_mv(data[trial][s]) for s in SENSORS}
    result += f"\n{'=' * 64}\nTRIAL {PERIODS[trial]}\n{'=' * 64}\n"

    # [1] fine carrier estimate (full-record FFT peak, ~0.04 Hz resolution vs Welch's 0.73 Hz bin).
    # An off-center LO parks the carrier on the bandpass skirt, converting phase wobble into fake AM.
    v = mv[SENSORS[0]] - mv[SENSORS[0]].mean()
    spec = np.abs(np.fft.rfft(v * np.hanning(len(v))))
    fr = np.fft.rfftfreq(len(v), 1 / Fs)
    fc_fine = fr[fr > 80][np.argmax(spec[fr > 80])]
    flag = "OK" if abs(fc_fine - CARRIER_FREQ) < 0.5 else "RECENTER LO"
    result += f"\n[1] carrier: FFT peak = {fc_fine:.3f} Hz, LO off by {CARRIER_FREQ - fc_fine:+.3f} Hz -> {flag}\n"

    z = {s: complex_baseband(mv[s], CARRIER_FREQ, BP_HALF_WIDTH) for s in SENSORS}
    tt = np.arange(len(z[SENSORS[0]])) / Fs

    # [2] clock lock: linear phase slope = constant LO offset (harmless); wander around that line
    # = LED clock drifting against the ADC clock (was 0.12 rad on two boards, 0.004 rad on one)
    result += "\n[2] clock lock (locked if phase wander << 0.1 rad):\n"
    slopes = []
    for s in SENSORS:
        ph = np.unwrap(np.angle(z[s]))
        fit = np.polyfit(tt, ph, 1)
        slopes.append(fit[0])
        wander = (ph - np.polyval(fit, tt)).std()
        verdict = "locked" if wander < 0.02 else ("marginal" if wander < 0.06 else "UNLOCKED - put carrier + ADC on one clock")
        result += f"{s:>10}: LO offset {fit[0] / (2 * np.pi):+.3f} Hz, phase wander {wander:.4f} rad -> {verdict}\n"

    zd = {s: z[s] * np.exp(-1j * np.median(slopes) * tt) for s in SENSORS}
    A = {s: 2 * np.abs(zd[s]) for s in SENSORS}

    # [3] AM/PM split: fluctuation along own carrier = AM, in quadrature = PM. Magnitude demod
    # only sees AM, so if PM dominates, the variance cell is analyzing the wrong axis.
    result += "\n[3] AM/PM split (envelope units, mV^2; Var_AM ~ Var(A_i) of the variance cell):\n"
    for s in SENSORS:
        c = zd[s].mean()
        dz = (zd[s] - c) * np.exp(-1j * np.angle(c))
        var_am, var_pm = 4 * np.var(dz.real), 4 * np.var(dz.imag)
        verdict = "AM dominates (model OK)" if var_am > var_pm else "PM DOMINATES (magnitude demod strips the common mode)"
        result += f"{s:>10}: a = {2 * abs(c):7.3f} mV, Var_AM = {var_am:.4f}, Var_PM = {var_pm:.4f} -> {verdict}\n"

    # [4] complex corr sees the full common mode (AM + PM); envelope corr only its AM part.
    # High |rho| with low/negative envelope corr = common mode is there but not on the amplitude axis.
    result += "\n[4] cross-channel correlation:\n"
    for i in range(len(SENSORS)):
        for j in range(i + 1, len(SENSORS)):
            si, sj = SENSORS[i], SENSORS[j]
            d1, d2 = zd[si] - zd[si].mean(), zd[sj] - zd[sj].mean()
            rho = np.mean(d1 * np.conj(d2)) / np.sqrt(np.mean(np.abs(d1) ** 2) * np.mean(np.abs(d2) ** 2))
            result += (f"{si:>10}-{sj:<10}: envelope corr {np.corrcoef(A[si], A[sj])[0, 1]:+.3f},  "
                       f"complex corr |rho| = {abs(rho):.3f} at {np.degrees(np.angle(rho)):+.1f} deg\n")

    # [5] additive electronics/pickup floor, measured where there is no carrier. Its envelope
    # contribution (4 * per-quadrature power) is amplitude-independent: if Var(n_i) from the
    # variance cell sits near this, "% indep" is the floor (an upper bound), not diode noise.
    result += f"\n[5] additive floor at {EMPTY_BAND_FREQ:.0f} Hz (carrier-free band):\n"
    ze = {s: complex_baseband(mv[s], EMPTY_BAND_FREQ, BP_HALF_WIDTH) for s in SENSORS}
    for s in SENSORS:
        s2 = (np.var(ze[s].real) + np.var(ze[s].imag)) / 2
        pred = 4 * s2
        result += f"{s:>10}: predicted floor part of Var(A) = {pred:.4f} mV^2  ({100 * pred / np.var(A[s]):.0f}% of measured Var(A))\n"
    e1, e2 = 2 * np.abs(ze[SENSORS[0]]), 2 * np.abs(ze[SENSORS[1]])
    result += f"floor envelope corr ({SENSORS[0]}-{SENSORS[1]}): {np.corrcoef(e1, e2)[0, 1]:+.3f}  (>> 0 = shared pickup, not independent noise)\n"

    # [6] Var(A) growing ~linearly with bandwidth = floor-dominated envelope. Very narrow bands
    # with an off-center LO fake a huge common AM (clock wobble on the filter skirt) - distrust them.
    result += f"\n[6] envelope corr vs bandpass half-width ({SENSORS[0]}-{SENSORS[1]}):\n"
    for hw in SWEEP_HALF_WIDTHS:
        z1 = complex_baseband(mv[SENSORS[0]], CARRIER_FREQ, hw)
        z2 = complex_baseband(mv[SENSORS[1]], CARRIER_FREQ, hw)
        A1, A2 = 2 * np.abs(z1), 2 * np.abs(z2)
        result += f"   +/-{hw:4.1f} Hz: corr {np.corrcoef(A1, A2)[0, 1]:+.3f}, Var(A_{SENSORS[0]}) = {A1.var():.4f} mV^2\n"

    # [7] common-mode rejection preview (first pair): d = A1/a1 - A2/a2 cancels m(t); what is left
    # is fractional private noise + floor. NOTE: with only 2 sensors Var(d) equals the variance-cell
    # split by algebra - it is a consistency check, not independent validation (3+ sensors are).
    A1, A2 = A[SENSORS[0]], A[SENSORS[1]]
    m_est = np.mean([A[s] / A[s].mean() for s in SENSORS], axis=0)
    d = A1 / A1.mean() - A2 / A2.mean()
    result += (f"\n[7] CMR: RMS(m_est) = {100 * m_est.std():.3f}%, RMS(d) = {100 * d.std():.3f}%, "
               f"corr(d, m_est) = {np.corrcoef(d, m_est)[0, 1]:+.3f}\n")
    result += (f"    variance below 0.5 Hz: m_est {100 * lowfreq_fraction(m_est, 0.5):.0f}%, "
               f"d {100 * lowfreq_fraction(d, 0.5):.0f}%  (high = slow drift, low = broadband)\n")

result += f"""
{'=' * 64}
INTERPRETATION
{'=' * 64}
- [2]/[3]: an unlocked clock or PM >> AM means the common mode is carrier phase jitter
  (LED clock vs ADC clock), not intensity fluctuation. 2*sqrt(I^2+Q^2) is blind to it, so the
  variance cell then correlates only the AM residue: low/negative corr and NaN var(m) follow.
  Fix: generate the carrier and sample on the same clock.
- [5]: if Var(n_i) from the variance cell ~= the floor prediction, "% indep" is really the
  electronics/pickup floor - an UPPER BOUND on private diode noise. Raise a_i (TIA / optical power).
- [6]: Var(A) scaling ~linearly with bandwidth = floor-dominated envelope; corr collapsing as the
  band widens means the common AM lives close to the carrier.
- [7]: RMS(d) << RMS(m_est) shows common-mode rejection working; d's floor sets the entropy budget.
"""

print(result)
savefolder_path.mkdir(parents=True, exist_ok=True)
with open(savefolder_path / "summary.txt", "w") as f:
    f.write(result + "\n")

if ser is not None and ser.is_open:
    ser.close()

## Online RLS

In [ ]:
ser = None
if not LOAD_FROM_FILE:
    ser = serial.Serial(PORT, BAUD, timeout=1)

LOAD_FILE_PATHS = [
    Path("rls_filter_data/07_02_2026/run_15_45_20/sample1.csv"),
    Path("rls_filter_data/07_02_2026/run_15_45_20/sample2.csv"),
    Path("rls_filter_data/07_02_2026/run_15_45_20/sample1.csv"),
    Path("rls_filter_data/07_02_2026/run_15_45_20/sample2.csv"),]

NUM_SAMPLES_TO_SAVE = 3000   # set to None to run until interrupted
now = datetime.now()
sample_count = 0
savefolder_path = Path(f"rls_filter_data/{now.strftime('%m_%d_%Y')}/run_{now.strftime('%H_%M_%S')}")
REF = "BPW34_1"
USABLE_SENSORS = [s for s in SENSORS if s != REF]

sos_bps = {s: signal.butter(4, [CARRIER_FREQ - 2, CARRIER_FREQ + 2], btype='bp', fs=Fs, output='sos') for s in SENSORS}
sos_lps = {s: signal.butter(4, 20, btype='low', fs=Fs, output='sos') for s in SENSORS}
zi_bps = {s: np.zeros((len(sos_bps[s]), 2)) for s in SENSORS} # type: ignore
zi_I  = {s: np.zeros((len(sos_lps[s]), 2)) for s in SENSORS} # type: ignore
zi_Q  = {s: np.zeros((len(sos_lps[s]), 2)) for s in SENSORS} # type: ignore
sample_num = 0

NTAPS = 1
filters  = {s: pa.filters.FilterRLS(n=NTAPS, mu=0.99999) for s in USABLE_SENSORS} # type: ignore
ref_hist = np.zeros(NTAPS)       # persistent rolling window of A[REF] across packets

def filter_pipeline(data):
    global sample_num, ref_hist
    N = SAMPLES_PER_PACKET
    t = np.arange(sample_num, sample_num + N) / Fs      # continuous LO phase
    lo_cos, lo_sin = np.cos(2*np.pi*CARRIER_FREQ*t), -np.sin(2*np.pi*CARRIER_FREQ*t)
    sample_num += N
 
    A = {}
    for s in SENSORS:
        v_mv = adc_to_mv(data[s])
        v_bp, zi_bps[s] = signal.sosfilt(sos_bps[s], v_mv, zi=zi_bps[s])
        I, zi_I[s] = signal.sosfilt(sos_lps[s], lo_cos*v_bp, zi=zi_I[s])
        Q, zi_Q[s] = signal.sosfilt(sos_lps[s], lo_sin*v_bp, zi=zi_Q[s])
        A[s] = 2.0 * np.sqrt(I**2 + Q**2)

    errors = {s: np.empty(N) for s in USABLE_SENSORS}
    ref = A[REF]
    for i in range(N):
        x = ref[i:i+1]                        # length-1 input window
        for s in USABLE_SENSORS:
            y = filters[s].predict(x)         # estimated common mode = w·A_REF
            filters[s].adapt(A[s][i], x)      # desired = sensor sample
            errors[s][i] = A[s][i] - y        # cleaned = sensor − common mode
    return errors

final_data = {s: [] for s in USABLE_SENSORS}

timeout = 0
while NUM_SAMPLES_TO_SAVE is None or sample_count < NUM_SAMPLES_TO_SAVE:
    if LOAD_FROM_FILE:
        data = reader()
        if data is None:  # reader is pacing itself against wall clock
            break
    else:
        data = try_read_realtime(ser)
        if data is not None:
            d = filter_pipeline(data)
            for s in USABLE_SENSORS:
                final_data[s].append(d[s])
        sample_count += SAMPLES_PER_PACKET
if ser is not None and ser.is_open:
    ser.close()


sig = {s: np.concatenate(final_data[s])[300:] for s in USABLE_SENSORS}


#save to csv
if not LOAD_FROM_FILE:
    savefolder_path.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame({s: sig[s] for s in USABLE_SENSORS})
    df.to_csv(savefolder_path / "rls_filtered_data.csv", index=False)
                
NFFT = 4096        
WINDOWSIZE = 2048
WINDOW = 'hamming'
OVERLAP = 90 

win = get_window(WINDOW, WINDOWSIZE)
noverlap = int(WINDOWSIZE * OVERLAP / 100)


with plt.style.context('dark_background'):
    fig, axes = plt.subplots(3, 1, figsize=(19, 21), sharex=False, constrained_layout=True)
    fig.suptitle(f'Channel spectrograms, Sampling Rate = {Fs:.2f} Hz', fontsize=30)

    for ax, s in zip(axes, USABLE_SENSORS):
        spec, freqs, t_bins, im = ax.specgram(sig[s], Fs=Fs, NFFT=WINDOWSIZE, window=win, noverlap=noverlap, pad_to=NFFT, cmap='magma')
        ax.set_ylabel(f'{s}\n(Hz)', fontsize = 30)
        ax.set_ylim(0, Fs/2)
        cbar = fig.colorbar(im, ax=ax, pad=0.01)
        cbar.set_label('Power (dB)', fontsize = 30)

    axes[-1].set_xlabel('Time (s)', fontsize = 30)
    plt.show()

    for s in USABLE_SENSORS:
        for s2 in USABLE_SENSORS:
            if s != s2:
                print(f"Correlation between {s} and {s2}: {np.corrcoef(sig[s], sig[s2])[0, 1]:.4f}")

if ser is not None and ser.is_open:
    ser.close()

## Noise Formulas

In [ ]:
# Photodiode + TIA noise budget.
# Every source is a POWER spectral density [A^2/Hz]; multiply by the noise bandwidth
# and sqrt for an RMS current, then x |Z_f| to refer it to the TIA output voltage.

import numpy as np

# ============================================================
# TUNABLE VALUES 
# ============================================================
# -- physical constants --
Q   = 1.602176634e-19   # elementary charge [C]
KB  = 1.380649e-23      # Boltzmann constant [J/K]
T   = 300.0             # temperature [K]

# -- optical / photodiode --
RESPONSIVITY = 0.55     # detector responsivity [A/W]  (BPW34 ~0.4 in the red)
P_OPTICAL    = 2.5e-6   # optical power on the diode [W]  -> sets the photocurrent
I_DARK       = 1.5e-9   # dark current [A]  (BPW34, low reverse bias)
C_DIODE      = 25e-12   # junction capacitance [F]  (BPW34 ~70pF @ 0V, ~25pF reverse-biased)

# -- TIA --
R_F  = 1.0e6            # feedback resistor = transimpedance gain [ohm]
C_F  = 1.0e-12          # feedback capacitor (stability pole) [F]
R_SH = 1.0e9            # photodiode shunt resistance [ohm]

# -- op-amp input noise (from its datasheet) --
E_N  = 8.0e-9           # input voltage noise density [V/sqrt(Hz)]
I_N  = 5.0e-15          # input current noise density [A/sqrt(Hz)]
C_IN = 5.0e-12          # amplifier input capacitance [F]

# -- measurement band --
F_EVAL  = 287.0         # frequency the noise is evaluated at (your carrier) [Hz]
BAND_HZ = 4.0           # noise bandwidth of the demod (+/-2 Hz -> 4 Hz)

# ============================================================
# DERIVED OPERATING POINT
# ============================================================
I_photo = RESPONSIVITY * P_OPTICAL
I_total = I_photo + I_DARK
C_tot   = C_DIODE + C_IN                        # total input-node capacitance
w       = 2 * np.pi * F_EVAL

Z_f   = R_F / (1 + 1j * w * R_F * C_F)          # feedback impedance  R_f || C_f
Z_s   = R_SH / (1 + 1j * w * R_SH * C_tot)      # source impedance    R_sh || C_tot
NG    = abs(1 + Z_f / Z_s)                      # noise gain seen by e_n
absZf = abs(Z_f)                                # transimpedance at F_EVAL [ohm]
V_T   = KB * T / Q                              # thermal voltage [V]

sqrtB = np.sqrt(BAND_HZ)
pA = lambda x: f"{x * 1e12:8.3f} pA/rtHz"       # current density
nV = lambda x: f"{x * 1e9:8.2f} nV/rtHz"        # voltage density
uV = lambda x: f"{x * 1e6:8.3f} uV rms"         # rms voltage over BAND_HZ

contrib = {}   # name -> output voltage density [V/rtHz]

def report_current_source(name, i_dens):
    """A white current noise that flows through the feedback impedance."""
    v_dens = i_dens * absZf
    contrib[name] = v_dens
    print(f"  input-referred : {pA(i_dens)}")
    print(f"  output density : {nV(v_dens)}   (x |Z_f| = {absZf/1e6:.3f} MOhm)")
    print(f"  output rms     : {uV(v_dens * sqrtB)}")

print("=" * 60)
print("OPERATING POINT")
print("=" * 60)
print(f"  I_photo        = {I_photo*1e6:.4f} uA   (R * P_opt)")
print(f"  I_dark         = {I_DARK*1e9:.4f} nA")
print(f"  I_total        = {I_total*1e6:.4f} uA")
print(f"  V_out (DC)     = I_total * R_f = {I_total*R_F:.4f} V   (check vs supply rails)")
print(f"  C_tot          = {C_tot*1e12:.1f} pF   (C_diode + C_in)")
print(f"  |Z_f| @ {F_EVAL:.0f} Hz = {absZf/1e6:.4f} MOhm   (transimpedance at the carrier)")
print(f"  thermal voltage kT/q = {V_T*1e3:.2f} mV")

print("\n" + "=" * 60)
print("STEP 1 - SHOT NOISE   S = 2q*I_total   (scales as sqrt(I))")
print("=" * 60)
i_shot = np.sqrt(2 * Q * I_total)
report_current_source("shot", i_shot)

print("\n" + "=" * 60)
print("STEP 2 - DARK-CURRENT SHOT   S = 2q*I_dark   (subset of step 1)")
print("=" * 60)
i_dark = np.sqrt(2 * Q * I_DARK)
print(f"  input-referred : {pA(i_dark)}   (flows even with the LED off)")

print("\n" + "=" * 60)
print("STEP 3 - FEEDBACK-RESISTOR THERMAL   S = 4kT/R_f   (amplitude-independent floor)")
print("=" * 60)
i_rf = np.sqrt(4 * KB * T / R_F)
report_current_source("Rf_thermal", i_rf)

print("\n" + "=" * 60)
print("STEP 4 - OP-AMP CURRENT NOISE   S = i_n^2")
print("=" * 60)
report_current_source("opamp_in", I_N)

print("\n" + "=" * 60)
print("STEP 5 - OP-AMP VOLTAGE NOISE   e_n * noise_gain   (the 'e_n*C' term)")
print("=" * 60)
v_en = E_N * NG
contrib["opamp_en"] = v_en
f_zero = 1 / (2 * np.pi * R_F * C_tot)  
f_pole = 1 / (2 * np.pi * R_F * C_F)  
print(f"  noise gain @ {F_EVAL:.0f} Hz = {NG:.3f}   (rises toward 1 + C_tot/C_f = {1 + C_tot/C_F:.1f})")
print(f"  e_n*C corner   : zero {f_zero:.1f} Hz,  pole {f_pole/1e3:.1f} kHz")
print(f"  output density : {nV(v_en)}")
print(f"  output rms     : {uV(v_en * sqrtB)}")
print(f"  (equiv input current e_n*2*pi*f*C_tot = {pA(E_N * w * C_tot)})")

print("\n" + "=" * 60)
print("STEP 6 - TOTAL NOISE (quadrature sum of the output densities)")
print("=" * 60)
v_tot = np.sqrt(sum(v**2 for v in contrib.values()))
v_tot_rms = v_tot * sqrtB
i_tot_in  = v_tot / absZf                 # input-referred total current density
NEP = i_tot_in / RESPONSIVITY             # noise-equivalent power [W/rtHz]
print(f"  total output density = {nV(v_tot)}")
print(f"  total output rms     = {uV(v_tot_rms)}   over {BAND_HZ:.0f} Hz")
print(f"  input-referred       = {pA(i_tot_in)}")
print(f"  NEP                  = {NEP*1e15:.2f} fW/rtHz")
print("  breakdown (% of noise VARIANCE):")
for name, v in sorted(contrib.items(), key=lambda kv: -kv[1]):
    print(f"     {name:<12}: {100 * (v**2) / (v_tot**2):5.1f} %")

print("\n" + "=" * 60)
print("STEP 7 - SIGNAL & SNR")
print("=" * 60)
v_sig = I_photo * absZf
print(f"  signal output    = I_photo * |Z_f| = {v_sig*1e3:.3f} mV")
print(f"  SNR (power)      = {10*np.log10((v_sig/v_tot_rms)**2):.1f} dB")
print(f"  SNR (amplitude)  = {v_sig / v_tot_rms:.0f} x")

print("\n" + "=" * 60)
print("STEP 8 - REGIME:  shot-limited vs thermal(floor)-limited")
print("=" * 60)
shot_over_thermal = (2 * Q * I_total) / (4 * KB * T / R_F)   # PSD ratio
print(f"  shot / Rf-thermal (power) = {shot_over_thermal:.2f}")
print(f"  crossover target : I_total * R_f > 2kT/q = {2*V_T*1e3:.1f} mV")
print(f"  you have         : I_total * R_f = {I_total*R_F*1e3:.1f} mV")
if I_total * R_F > 2 * V_T:
    print("  -> SHOT-LIMITED: Var(n) scales with optical power.")
else:
    print("  -> THERMAL/ELECTRONIC-LIMITED: Var(n) is fixed (the additive floor you see now).")
    print(f"     raise I_photo or R_f by {2*V_T/(I_total*R_F):.1f}x to cross into shot-limited.")


## Averaging

In [ ]:
ser = None
if not LOAD_FROM_FILE:
    ser = serial.Serial(PORT, BAUD, timeout=1)

LOAD_FILE_PATHS = []

NUM_SAMPLES_TO_SAVE = 1000  # set to None to run until interrupted
now = datetime.now()
sample_count = 0
savefolder_path = Path(f"averaging_data/{now.strftime('%m_%d_%Y')}/run_{now.strftime('%H_%M_%S')}")

sos_bps = {s: signal.butter(4, [CARRIER_FREQ - 2, CARRIER_FREQ + 2], btype='bp', fs=Fs, output='sos') for s in SENSORS}
sos_lps = {s: signal.butter(4, 150, btype='low', fs=Fs, output='sos') for s in SENSORS}
zi_bps = {s: np.zeros((len(sos_bps[s]), 2)) for s in SENSORS} # type: ignore
zi_I  = {s: np.zeros((len(sos_lps[s]), 2)) for s in SENSORS} # type: ignore



def filter_pipeline(data):
    global sample_num
    N = SAMPLES_PER_PACKET
    t = np.arange(sample_num, sample_num + N) / Fs      # continuous LO phase
    lo_cos, lo_sin = np.cos(2*np.pi*CARRIER_FREQ*t), -np.sin(2*np.pi*CARRIER_FREQ*t)
    sample_num += N
 
    A = {}
    for s in SENSORS:
        v_mv = adc_to_mv(data[s])
        v_bp, zi_bps[s] = signal.sosfilt(sos_bps[s], v_mv, zi=zi_bps[s])
        I, zi_I[s] = signal.sosfilt(sos_lps[s], lo_cos*v_bp, zi=zi_I[s])
        A[s] = 2.0 * I
    return A
    

sample_num = 0
timeout = 0
final_data = {s: [] for s in SENSORS}

while NUM_SAMPLES_TO_SAVE is None or sample_count < NUM_SAMPLES_TO_SAVE:
    if LOAD_FROM_FILE:
        data = reader()
        if data is None:  # reader is pacing itself against wall clock
            break
    else:
        data = try_read_realtime(ser)
        if data is not None:
            d = filter_pipeline(data)
            for s in SENSORS:
                final_data[s].append(d[s])
        sample_count += SAMPLES_PER_PACKET
if ser is not None and ser.is_open:
    ser.close()


sig = {s: np.concatenate(final_data[s])[300:] for s in SENSORS}

means = { s: np.mean(sig[s]) for s in SENSORS }

bits = []
for s in SENSORS:
    for s2 in SENSORS:
        if s != s2:
            continue

#save to csv
if not LOAD_FROM_FILE:
    savefolder_path.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame({s: sig[s] for s in USABLE_SENSORS})
    df.to_csv(savefolder_path / "rls_filtered_data.csv", index=False)
                
if ser is not None and ser.is_open:
    ser.close()